In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *

In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_720s.20241120_000000_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz']

In [227]:
file_hmi = files[0]
file_fdt = files[1]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

data_hmi = rebin(data_hmi, 8, update_header=header_hmi)
data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=True, mu_thr=0.2, crota=header_fdt['CROTA'] + 0.2)


data_fdt[np.abs(data_fdt) < 1e-3] = np.nan
data_fdt[np.isnan(data_hmi)] = np.nan
data_hmi[np.isnan(data_fdt)] = np.nan

In [228]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-50, vmax=50)

In [229]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, cmap='seismic', vmin=-50, vmax=50)

In [230]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.01)
plt.xlim(-200,200)
plt.ylim(-200,200)

(-200.0, 200.0)

In [222]:
dx = 0.5

x = []
y = []
for x_ in np.arange(-10,10,dx):
    a = x_ ** 3
    b = (x_ + dx) ** 3

    t = np.where(np.all([data_hmi > a, data_hmi < b], axis=0))
    x += [np.mean(data_hmi[t])]
    y += [np.mean(data_fdt[t])]

x = np.array(x)
y = np.array(y)

plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.01)
plt.plot(x, y, color='black')

plt.xlim(-50,50)
plt.ylim(-50,50)

#plt.grid(True)
plt.tight_layout()

In [206]:
t = np.where(np.abs(data_hmi) < 10)

np.polyfit(data_hmi[t], data_fdt[t], 1)

array([ 0.63351208, -0.29203074])

In [124]:
?np.polyfit

In [68]:
header_fdt['RSUN_ARC'] / header_fdt['CDELT1']

329.6575871413028

In [72]:
header_hmi['RSUN_OBS'] / header_hmi['CDELT1']

321.1803014216096